# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/rithy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/rithy/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '3aede8'. Skipping!
Property 'summary' already exists in node 'e3b5eb'. Skipping!
Property 'summary' already exists in node '3433f5'. Skipping!
Property 'summary' already exists in node 'b65d5d'. Skipping!
Property 'summary' already exists in node 'e0c27d'. Skipping!
Property 'summary' already exists in node 'ae416b'. Skipping!
Property 'summary' already exists in node 'a01e12'. Skipping!
Property 'summary' already exists in node 'f94cf1'. Skipping!
Property 'summary' already exists in node '9c4e76'. Skipping!
Property 'summary' already exists in node '87d9fe'. Skipping!
Property 'summary' already exists in node '345032'. Skipping!
Property 'summary' already exists in node '2f53b6'. Skipping!
Property 'summary' already exists in node '5c5693'. Skipping!
Property 'summary' already exists in node 'ace788'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/41 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'f94cf1'. Skipping!
Property 'summary_embedding' already exists in node 'e3b5eb'. Skipping!
Property 'summary_embedding' already exists in node '5c5693'. Skipping!
Property 'summary_embedding' already exists in node '3aede8'. Skipping!
Property 'summary_embedding' already exists in node 'ae416b'. Skipping!
Property 'summary_embedding' already exists in node '3433f5'. Skipping!
Property 'summary_embedding' already exists in node 'b65d5d'. Skipping!
Property 'summary_embedding' already exists in node '2f53b6'. Skipping!
Property 'summary_embedding' already exists in node '9c4e76'. Skipping!
Property 'summary_embedding' already exists in node '345032'. Skipping!
Property 'summary_embedding' already exists in node '87d9fe'. Skipping!
Property 'summary_embedding' already exists in node 'a01e12'. Skipping!
Property 'summary_embedding' already exists in node 'e0c27d'. Skipping!
Property 'summary_embedding' already exists in node 'ace788'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 39, relationships: 478)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 39, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

SingleHopeSpecificQuerySynthezier - generates straightforward and fact-based questions that be answered by looking at a single piece of information

MultiHopAbstractQuerySynthesizer - generates complex/open-ended questions that require connecting multiple pieces of information and/or some reasoning/summarization.

MultiHopSpecificQuerySynthesizer - generates specific questions that require combining information from multiple parts of the data




Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What information does Volume 4 provide about t...,[Direct Loan Eligibility After an Enrollment S...,Volume 4 explains that if a student who receiv...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 685.303(b)(3)(iv) about in rela...,[Direct Loan Disbursements to Students Who Tem...,34 CFR 685.303(b)(3)(iv) pertains to direct lo...,single_hop_specifc_query_synthesizer
2,What is exit counseling in the context of Dire...,[Chapter 2 Direct Loan Counseling Counseling O...,Exit counseling is a requirement for student a...,single_hop_specifc_query_synthesizer
3,Whaat is a Fedral Stafford Loan and how does i...,[Entrance Counseling Entrance counseling is re...,A Federal Stafford Loan is a type of federal s...,single_hop_specifc_query_synthesizer
4,What is the HEA and how does it relate to stud...,[provided by those companies are already offer...,"The HEA, or Higher Education Act, requires the...",single_hop_specifc_query_synthesizer
5,What are the different loan types involved in ...,[<1-hop>\n\nChapter 2 Direct Loan Counseling C...,The loan types involved in Direct Loan counsel...,multi_hop_abstract_query_synthesizer
6,What is the role of direct loan exit counselin...,[<1-hop>\n\nprovided by those companies are al...,Direct loan exit counseling is designed to pro...,multi_hop_abstract_query_synthesizer
7,Wht documentation of enrollment and costs is n...,[<1-hop>\n\nDirect Loan Eligibility After an E...,"According to the context, when a student who r...",multi_hop_abstract_query_synthesizer
8,How does Title IV regulation impact the requir...,[<1-hop>\n\nprovided by those companies are al...,Title IV regulations require that schools prov...,multi_hop_specific_query_synthesizer
9,How can a student find info on student aid at ...,[<1-hop>\n\nprovided by those companies are al...,A student can find information about federal s...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/30 [00:00<?, ?it/s]

Property 'summary' already exists in node '087161'. Skipping!
Property 'summary' already exists in node '269ab4'. Skipping!
Property 'summary' already exists in node 'f74e06'. Skipping!
Property 'summary' already exists in node '16beae'. Skipping!
Property 'summary' already exists in node '958f13'. Skipping!
Property 'summary' already exists in node '8e87ef'. Skipping!
Property 'summary' already exists in node '341051'. Skipping!
Property 'summary' already exists in node 'a93457'. Skipping!
Property 'summary' already exists in node 'ce34ee'. Skipping!
Property 'summary' already exists in node 'e44c1b'. Skipping!
Property 'summary' already exists in node '9e0e58'. Skipping!
Property 'summary' already exists in node 'e4d109'. Skipping!
Property 'summary' already exists in node 'f0e069'. Skipping!


...

Property 'summary_embedding' already exists in node '087161'. Skipping!
Property 'summary_embedding' already exists in node 'f74e06'. Skipping!
Property 'summary_embedding' already exists in node 'a93457'. Skipping!
Property 'summary_embedding' already exists in node 'e44c1b'. Skipping!
Property 'summary_embedding' already exists in node '269ab4'. Skipping!
Property 'summary_embedding' already exists in node '16beae'. Skipping!
Property 'summary_embedding' already exists in node 'f0e069'. Skipping!
Property 'summary_embedding' already exists in node '958f13'. Skipping!
Property 'summary_embedding' already exists in node '8e87ef'. Skipping!
Property 'summary_embedding' already exists in node 'ce34ee'. Skipping!
Property 'summary_embedding' already exists in node '341051'. Skipping!
Property 'summary_embedding' already exists in node '9e0e58'. Skipping!
Property 'summary_embedding' already exists in node 'e4d109'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of Chapter 3 in the c...,[Direct Unsubsidized Loans for Students Whose ...,Chapter 3 discusses Direct Unsubsidized Loans ...,single_hop_specifc_query_synthesizer
1,What does Title IV specify regarding Direct Lo...,[Direct Loan Eligibility After an Enrollment S...,"Title IV program funds, including Direct Loans...",single_hop_specifc_query_synthesizer
2,What is 34 CFR 685.303(b)(3)(iv) about?,[Direct Loan Disbursements to Students Who Tem...,34 CFR 685.303(b)(3)(iv) pertains to direct lo...,single_hop_specifc_query_synthesizer
3,What are the counseling requirements for a stu...,[Chapter 2 Direct Loan Counseling Counseling O...,There is a special counseling requirement for ...,single_hop_specifc_query_synthesizer
4,Is a proffesional judgment decicion made for l...,[<1-hop>\n\nDirect Unsubsidized Loans for Stud...,"Yes, a professional judgment decision can be m...",multi_hop_abstract_query_synthesizer
5,Considering the restrictions on other Title IV...,[<1-hop>\n\nDirect Unsubsidized Loans for Stud...,When parents refuse to complete the FAFSA or e...,multi_hop_abstract_query_synthesizer
6,Hwo does the temme of 'Temporary Cease of Half...,[<1-hop>\n\nDirect Loan Eligibility After an E...,"According to the provided context, if a studen...",multi_hop_abstract_query_synthesizer
7,What is the role of the FFEL Program in the en...,[<1-hop>\n\nEntrance Counseling Entrance couns...,The FFEL Program provided federal loans like F...,multi_hop_abstract_query_synthesizer
8,How do Chapters 2 and 5 relate to the counseli...,[<1-hop>\n\nDirect Unsubsidized Loans for Stud...,Chapter 2 discusses the counseling requirement...,multi_hop_specific_query_synthesizer
9,Hwo Title IV is relted to the loan eligibilty ...,[<1-hop>\n\nprovided by those companies are al...,The context explains that Title IV program fun...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data - Rithvik"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (also called student Federal PLUS Loans)  \n\nAdditionally, Subsidized and Unsubsidized Federal Stafford Loans, Federal SLS Loans, and Federal PLUS Loans were made under the Federal Family Education Loan (FFEL) Program before new loans under that program ended effective July 1, 2010.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

##### ✅ Answer:

- `qa_evaluator`: Evaluates if the model's answer is correct compared to reference answer
- `labeled_helpfulness_evaluator`: Evaluates how helpful the answer is to the user (how useful, relevant, etc.) based on the correct answer
- `empathy_evaluator`: Evaluates whether the response makes the user feel heard and understood.

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'kind-spark-21' at:
https://smith.langchain.com/o/7ce39702-5029-477f-bda3-10342686da6c/datasets/3fd743ae-ad37-4ef6-87c6-4f7ef660c62d/compare?selectedSessions=d8fe2b4d-310b-4a88-bf28-4cd58bf0d2aa




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How does Chapter 3 expand on the counseling re...,I don't know.,None,Chapter 2 outlines the general counseling requ...,0,0,0,0.709559,270005bd-300a-4fdb-b96a-1b7ddc1e1885,76859f0c-3cfe-426e-98e5-619417f1cc32
1,How do the counseling requirements for Direct ...,Based on the provided context:\n\nChapter 2 ou...,None,Chapter 2 outlines that first-time student bor...,1,1,0,5.145195,be2f4544-af21-487d-917f-db941a1a314f,72f0e893-12b3-46c4-9011-e4a5ac85d567
2,How does FAFSA information influence the eligi...,Based on the provided context:\n\n- If the par...,None,When parents refuse to complete the FAFSA form...,1,1,0,3.276923,00d290ec-84ad-4a60-bbc8-a3461638c107,b32e44fa-6aba-47fd-8d4c-383352196359
3,What are the requierd content for exit counsel...,I don't know.,None,34 CFR 685.303(b)(3)(iv) addresses direct loan...,0,0,0,0.482526,866d36a8-7ee5-46bb-ad8e-3d2352ae75a0,2cd1bf2e-ed50-4d5b-9a0f-6f5a92c0cb50
4,How does the legal adoption of a parent affect...,The legal adoption of a parent allows that par...,None,"According to the context, the definition of <P...",1,0,0,1.333423,3b4409b5-e70a-4925-8f5b-ecc67412be0a,1ea4b547-84bc-4c48-bbb2-0dde8d81dbcc
5,H0w do the rules about Tmeporary Cease of Half...,Based on the context provided:\n\nIf a student...,None,"According to the provided context, if a studen...",1,1,0,4.385722,8fd707a1-ec3e-4307-b162-3fd6754d79c1,eac27fb0-82bb-44d6-9909-304d7c067072
6,How does documentation of counseling completio...,According to the Department's guidelines in th...,None,"Documentation of counseling completion, such a...",1,1,0,5.654682,bf9bbd2b-103f-4d04-af26-615e2354135c,33321a03-2bac-4ded-97e5-20d9ce8e585f
7,Whatt is the importnce of exit counseling (exi...,Exit counseling is important in the loan proce...,None,Exit counseling is crucial because it provides...,1,1,0,3.019946,9f413cb9-60e9-4060-bc20-771a9dc1027e,3c896cc8-1527-475f-9e87-7dbcaee29489
8,What does 34 CFR 685.303(b)(3)(iv) cover regar...,I don't know.,None,34 CFR 685.303(b)(3)(iv) pertains to direct lo...,0,0,0,0.481257,58b3032e-9513-49af-8fda-17fa9a66f643,df859368-f38f-469c-9990-3936e5e05651
9,What is discussed in Volume 4 regarding Direct...,"Based on the provided context, Volume 4, Chapt...",None,Volume 4 covers the rules for Direct Loan elig...,0,0,0,1.019714,d013849b-2851-4098-9034-2a21a1d18819,8708b217-5b24-4e44-a8ea-87d85988bf82


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:
Larger chunks would mean there is more context in a single chunk for the model to work with. The con here is that there may be irrelevant information in that chunk, since it is larger.


In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Using a better embedding model would yield more accurate understanding of text, which would improve retrieval and overall performance.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided in the context, there are several types of loans available to students and their parents:\n\n1. **Direct Subsidized Loans:** These loans are based on the student's financial need and have interest subsidized by the government while the student is in school.\n\n2. **Direct Unsubsidized Loans:** These are available regardless of financial need and interest accrues while the student is in school.\n\n3. **Direct PLUS Loans:** These loans are available to parents of dependent students (Direct PLUS Loan for parents) or to graduate/professional students (Direct PLUS Loan for students). They can cover the student's cost of attendance (COA) minus other financial aid, with no fixed loan limit, but the loan can't exceed the COA less other aid.\n\n4. **Additional Unsubsidized Loans:** If the parent of a dependent student is unable to obtain a Direct PLUS Loan, or if the student is independent, they may be eligible for additional Direc

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'kind-tub-29' at:
https://smith.langchain.com/o/7ce39702-5029-477f-bda3-10342686da6c/datasets/3fd743ae-ad37-4ef6-87c6-4f7ef660c62d/compare?selectedSessions=000ef2c9-0f43-4b84-b8a4-c86a4c7ff251




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How does Chapter 3 expand on the counseling re...,Thank you for your thoughtful question. I unde...,None,Chapter 2 outlines the general counseling requ...,0,0,1,3.442517,270005bd-300a-4fdb-b96a-1b7ddc1e1885,82859853-1551-42b7-bf28-c4402d9e7314
1,How do the counseling requirements for Direct ...,Thank you for your thoughtful question. It rea...,None,Chapter 2 outlines that first-time student bor...,1,1,1,7.248158,be2f4544-af21-487d-917f-db941a1a314f,5924e364-03c3-4aba-8f2c-2d667a9d1be5
2,How does FAFSA information influence the eligi...,Thank you for your thoughtful question. It sou...,None,When parents refuse to complete the FAFSA form...,1,0,1,7.581853,00d290ec-84ad-4a60-bbc8-a3461638c107,c1baed42-6855-43a3-9dbc-ab224da7f2f9
3,What are the requierd content for exit counsel...,Thank you for your thoughtful question. From t...,None,34 CFR 685.303(b)(3)(iv) addresses direct loan...,0,0,1,12.760947,866d36a8-7ee5-46bb-ad8e-3d2352ae75a0,e51be6c0-f2fc-4025-a938-e31d725247c0
4,How does the legal adoption of a parent affect...,Thank you for your thoughtful question. Based ...,None,"According to the context, the definition of <P...",1,0,1,2.569132,3b4409b5-e70a-4925-8f5b-ecc67412be0a,b6681516-163d-4356-90b8-f51196b10e5b
5,H0w do the rules about Tmeporary Cease of Half...,I understand how important it is to have clari...,None,"According to the provided context, if a studen...",1,1,1,6.820835,8fd707a1-ec3e-4307-b162-3fd6754d79c1,ffb58672-23cc-42ee-97ca-551bbf301763
6,How does documentation of counseling completio...,Thank you for your thoughtful question. It sou...,None,"Documentation of counseling completion, such a...",1,1,1,5.836602,bf9bbd2b-103f-4d04-af26-615e2354135c,eabbd95e-a017-40ea-8360-e7d54568b167
7,Whatt is the importnce of exit counseling (exi...,Thank you for your thoughtful question. Exit a...,None,Exit counseling is crucial because it provides...,1,1,1,4.499283,9f413cb9-60e9-4060-bc20-771a9dc1027e,37e1465e-6d0a-4733-b0dd-0a61b3c1eda3
8,What does 34 CFR 685.303(b)(3)(iv) cover regar...,Thank you for your thoughtful question. I trul...,None,34 CFR 685.303(b)(3)(iv) pertains to direct lo...,0,0,1,3.901950,58b3032e-9513-49af-8fda-17fa9a66f643,ecc0dcaf-035e-41e0-9ad0-3cdc78546176
9,What is discussed in Volume 4 regarding Direct...,Thank you for your thoughtful question. Based ...,None,Volume 4 covers the rules for Direct Loan elig...,0,0,1,2.516234,d013849b-2851-4098-9034-2a21a1d18819,c57f2637-7c67-46d7-8a2c-b1588263f5fa


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

![First Chain](first.jpg)
![Second Chain](second.jpg)


Differences:

Empathy - Our RAG prompt for the second chain included "You must answer the question using empathy and kindness, and make sure the user feels heard." which bumped up the empathy scores on the second chain.

Latency - Latency went up by 3 seconds. I believe with larger chunks, the model would need to surf through more text to find the correct answer to the questions given.

Helpfulness - went down by a significant amount. I believe this went down because of the empathy aspect. Even though our RAG prompt still said to say "I don't know" if the model does not know the answer, the empathy prompt made the model produce a longwinded response just to essentially say it does not know the answer.



